In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/logs_dataset (1).csv")
df.head()


,@timestamp,_id,ip_address
0,"July 8th 2019, 14:43:03.000",XswJ0msBoTGddM7vxMDB,10.1.1.285
1,"July 8th 2019, 14:43:01.000",dKQJ0msB7mP0GwVzvJjz,10.1.2.389
2,"July 8th 2019, 14:42:59.000",CcwJ0msBoTGddM7vtb8y,10.1.1.415
3,"July 8th 2019, 14:42:57.000",bKQJ0msB7mP0GwVzrZdT,10.1.1.79
4,"July 8th 2019, 14:42:55.000",L6QJ0msB7mP0GwVzpZeI,10.1.1.60


In [ ]:
df.describe()

,@timestamp,_id,ip_address
count,339347,339347,339346
unique,339293,335107,386
top,"July 4th 2019, 09:27:05.000",#NAME?,10.1.1.63
freq,7,4241,2137


In [ ]:
df['@timestamp'] = (
    df['@timestamp']
    .str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)
)
df['@timestamp'] = pd.to_datetime(
    df['@timestamp'],
    format='%B %d %Y, %H:%M:%S.%f'
)

In [ ]:
df.sort_values(['ip_address','@timestamp'],inplace=True)

In [ ]:
df['shift_time'] = df.groupby(['ip_address'])['@timestamp'].shift(1)

In [ ]:
df.head()

,@timestamp,_id,ip_address,shift_time
338337,2019-06-28 19:07:40,IFh8n2sB7mP0GwVzcEJN,10.1.1.1,NaT
338283,2019-06-28 19:09:57,aoB-n2sBoTGddM7vh6Ft,10.1.1.1,2019-06-28 19:07:40
338219,2019-06-28 19:12:37,toCAn2sBoTGddM7v-MNt,10.1.1.1,2019-06-28 19:09:57
338118,2019-06-28 19:16:49,RliEn2sB7mP0GwVz0L3b,10.1.1.1,2019-06-28 19:12:37
336582,2019-06-28 20:20:49,F4S_n2sBoTGddM7vaHTR,10.1.1.1,2019-06-28 19:16:49


In [ ]:
df['time_diff']=(df['@timestamp']-df['shift_time']).dt.seconds//60

In [ ]:
df['date']=df['@timestamp'].dt.date

In [ ]:
df['date']=df['@timestamp'].dt.weekday

In [ ]:
df['hour']=df['@timestamp'].dt.hour

In [ ]:
df['is_weekend'] = ((df['@timestamp'].dt.weekday == 5) | (df['@timestamp'].dt.weekday == 6)).astype(int)

In [ ]:
df['hour_bucket'] = df['hour']//4
# This integer division by 4, so we can analyse 4-hour buckets instead of every single hour

In [ ]:
ip_addr ='ip_address'

In [ ]:
ip_counts=df.groupby(ip_addr)['@timestamp'].count().reset_index()

In [ ]:
ip_counts.head()

,ip_address,@timestamp
0,10.1.1.1,699
1,10.1.1.100,1377
2,10.1.1.101,687
3,10.1.1.106,671
4,10.1.1.109,655


In [ ]:
ip_counts = ip_counts.rename(columns={'@timestamp':'total_count'})
daily_counts = df.groupby([ip_addr,'date'])['@timestamp'].count().reset_index()
daily_counts = daily_counts.rename(columns={'@timestamp':'daily_counts'})
daily_counts_avg = daily_counts.groupby(ip_addr).daily_counts.median().reset_index()
daily_counts_avg.head(5)
weekend_counts = df.groupby([ip_addr, 'is_weekend'])['@timestamp'].count().reset_index()
weekend_counts

,ip_address,is_weekend,@timestamp
0,10.1.1.1,0,418
1,10.1.1.1,1,281
2,10.1.1.100,0,784
3,10.1.1.100,1,593
4,10.1.1.101,0,408
...,...,...,...
767,10.1.2.90,1,535
768,10.1.2.95,0,798
769,10.1.2.95,1,564
770,10.1.2.99,0,418
